In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.feature_selection import VarianceThreshold
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    adjusted_rand_score,
    normalized_mutual_info_score,
    accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report,
    silhouette_score
)

df = pd.read_csv("battery_observations.csv", index_col=0)
meta = pd.read_csv("battery_metadata.csv", index_col=0)
df2 = df.join(meta[["climate_zone", "chemistry"]])
df2 = df2.sort_values(["battery_id", "cycle_index"])
df2 = df2.reset_index()
meta = meta.reset_index()

g = lambda x: int(x.split('-')[-1])
df2["battery_id"] = df2["battery_id"].apply(g)
meta["battery_id"] = meta["battery_id"].apply(g)
battery_ids = list(set(df2["battery_id"]))
materials = list(set(meta["chemistry"]))
climate_zones = list(set(meta["climate_zone"]))
labels = list(set(df2["degradation_status"]))

train_labels = pd.read_csv("train_batteries.csv")["battery_id"].apply(g)
val_labels = pd.read_csv("validation_batteries.csv")["battery_id"].apply(g)

train = df2[df2["battery_id"].isin(train_labels)]
val = df2[df2["battery_id"].isin(val_labels)]

In [2]:

cycle_features = [
    "soc_start_pct",
    "soc_end_pct",
    "depth_of_discharge_pct",
    "high_soc_exposure_fraction",
    "low_soc_exposure_fraction",
    "charge_c_rate",
    "discharge_c_rate",
    "charge_current_a",
    "discharge_current_a",
    "charge_duration_min",
    "discharge_duration_min",
    "energy_discharged_kwh",
    "ambient_temp_c",
    #"pack_temp_c",
    "cooling_active",
    "pack_voltage_mean_v"
]

extended_features = cycle_features + [
    "battery_age_days",
    "time_since_previous_cycle_hours",
    "equivalent_full_cycles",
    "cumulative_high_temp_hours",
    "pack_voltage_charge_v",
    "pack_voltage_discharge_v"
]

features_A = [
    "battery_age_days",
    "equivalent_full_cycles",
]

features_B = features_A + [
    "internal_resistance_mOhm",
]

features_C = features_B + [
    "cumulative_high_temp_hours",
]


In [17]:
class_names = ["Normal", "Moderate", "Severe", "Critical"]
class_names_mapping = {class_names[i]:i for i in range(len(class_names))}

def perform_logistic_regression(X_train, X_val, y_train, y_val, penalty):
    logreg = LogisticRegression(max_iter=2000, class_weight="balanced", l1_ratio=0 if "l2" == penalty else 1, solver="saga")
    logreg.fit(X_train, y_train)
    predicted = logreg.predict(X_val) # is list of strings

    predicted_num = [class_names_mapping[x] for x in predicted]
    y_val_num = [class_names_mapping[x] for x in y_val]

    metrics = {
        "ARI": adjusted_rand_score(y_val_num, predicted_num),
        "NMI": normalized_mutual_info_score(y_val_num, predicted_num),
        "accuracy": accuracy_score(y_val_num, predicted_num),
        "macro_F1": f1_score(y_val, predicted, labels=class_names,
                                average="macro", zero_division=0),
    }

    return metrics, predicted


In [18]:
preprocess = Pipeline([
    ("scaler", StandardScaler()),
    ("variance", VarianceThreshold(threshold=0.0))
])

use_features = features_C

X_train = preprocess.fit_transform(train[use_features])
X_val = preprocess.transform(val[use_features])
y_train = train["degradation_status"].to_numpy()
y_val = val["degradation_status"].to_numpy()

metrics_L2, predicted_L2 = perform_logistic_regression(X_train, X_val, y_train, y_val, "l2")
metrics_L1, predicted_L1 = perform_logistic_regression(X_train, X_val, y_train, y_val, "l1")
print('metrics L2 error:'); print(metrics_L2)
print('metrics L1 error:'); print(metrics_L1)

confusion_df_L2 = pd.DataFrame(
    confusion_matrix(
        predicted_L2,
        y_val,
        labels=class_names
    ),
    index=[f"Actual {label}" for label in class_names],
    columns=[f"Predicted {label}" for label in class_names]
    )

confusion_df_L1 = pd.DataFrame(
    confusion_matrix(
        predicted_L1,
        y_val,
        labels=class_names
    ),
    index=[f"Actual {label}" for label in class_names],
    columns=[f"Predicted {label}" for label in class_names]
    )

print("\nValidation confusion matrix (L2 error):")
print(confusion_df_L2)

print("\nValidation confusion matrix (L1 error):")
print(confusion_df_L1)

print("\nValidation classification report (L2 error):")
print(classification_report(
    y_val,
    predicted_L2,
    labels=class_names,
    zero_division=0
))

print("\nValidation classification report (L1 error):")
print(classification_report(
    y_val,
    predicted_L1,
    labels=class_names,
    zero_division=0
))



/home/tobiasocula/Documents/ML/assignment/venv/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
/home/tobiasocula/Documents/ML/assignment/venv/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


metrics L2 error:
{'ARI': 0.2842447049634103, 'NMI': 0.30099802629572603, 'accuracy': 0.4583766909469303, 'macro_F1': 0.3903642665499839}
metrics L1 error:
{'ARI': 0.3583229210876403, 'NMI': 0.3456092687495236, 'accuracy': 0.5038421516048988, 'macro_F1': 0.416523072251383}

Validation confusion matrix (L2 error):
                 Predicted Normal  Predicted Moderate  Predicted Severe  \
Actual Normal                6947                1806                61   
Actual Moderate              1327                2392              1337   
Actual Severe                   0                 354              2098   
Actual Critical               364                3995              4278   

                 Predicted Critical  
Actual Normal                     0  
Actual Moderate                   5  
Actual Severe                     6  
Actual Critical                  16  

Validation confusion matrix (L1 error):
                 Predicted Normal  Predicted Moderate  Predicted Severe  \
Act